# 05 — Operating Condition Identification

**Goals**
- Q9: identify downhole operating regimes; which algorithms?
- Q10: time structure and regularities; interpret changes in downhole conditions

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from well_analysis.data import load_test_data
from well_analysis.signal import check_even_sampling
from well_analysis.signal.integration import integrate_acceleration, estimate_gravity_offset
from well_analysis.detection import detect_well_state
from well_analysis.analysis.dynamometer import estimate_pump_frequency, extract_dynamometer_cards
from well_analysis.analysis.clustering import (
    extract_card_features, cluster_operating_conditions, reduce_for_viz
)
from well_analysis.viz import plot_cluster_scatter

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

df = load_test_data()
_, fs = check_even_sampling(df['Timestamp'])
accel = df['Acceleration'].values
load  = df['Load'].values

is_running = detect_well_state(accel, fs=fs)
g_offset   = estimate_gravity_offset(accel, is_running)
_, position = integrate_acceleration(accel, fs=fs, gravity_offset=g_offset)
pump_freq  = estimate_pump_frequency(accel[is_running], fs=fs)

## 1. Approach (Q9)

This is an **unsupervised clustering** problem — no labelled ground truth.

**Pipeline**:
1. Extract one dynamometer card per pump cycle.
2. Compute a feature vector per card (area, load range, upstroke/downstroke asymmetry, …).
3. Normalise features and apply clustering.

**Algorithm choices**:
| Algorithm | When to use |
|-----------|-------------|
| **KMeans** | Fixed number of regimes, fast, interpretable centroids |
| **HDBSCAN** | Unknown number of clusters, handles noise/outliers |
| **GMM** | Soft cluster membership, elliptical clusters |
| **Hierarchical** | Dendrogram for regime hierarchy exploration |

We start with KMeans (interpretable) and validate with silhouette score.

In [ ]:
# Extract cards across all running data
cards = extract_dynamometer_cards(position[is_running], load[is_running], fs=fs, pump_freq=pump_freq)
print(f"Total cards extracted: {len(cards)}")

In [ ]:
features = extract_card_features(cards)
print(f"Feature matrix shape: {features.shape}")

In [ ]:
# Choose n_clusters — use elbow / silhouette to validate
from sklearn.metrics import silhouette_score

scores = {}
for k in range(2, 8):
    labels, _, scaler = cluster_operating_conditions(features, n_clusters=k)
    scores[k] = silhouette_score(features, labels)

best_k = max(scores, key=scores.get)
print('Silhouette scores:', {k: f"{v:.3f}" for k, v in scores.items()})
print(f"Best k: {best_k}")

In [ ]:
labels, kmeans, scaler = cluster_operating_conditions(features, n_clusters=best_k)
coords_2d = reduce_for_viz(features, scaler)

fig, ax = plt.subplots(figsize=(7, 5))
plot_cluster_scatter(coords_2d, labels, ax=ax)
plt.tight_layout()
plt.show()

## 2. Time structure of regimes (Q10)

In [ ]:
# Map each card back to its timestamp (start sample → timestamp)
# Note: card indices are relative to is_running mask — need to re-map
running_indices = np.where(is_running)[0]

card_times = []
for card in cards:
    global_start = running_indices[card['start']] if card['start'] < len(running_indices) else running_indices[-1]
    card_times.append(df['Timestamp'].iloc[global_start])

card_times = pd.Series(card_times)

fig, ax = plt.subplots(figsize=(14, 3))
scatter = ax.scatter(card_times, labels, c=labels, cmap='tab10', s=4, alpha=0.7)
ax.set_xlabel('Time')
ax.set_ylabel('Cluster')
ax.set_title('Operating regime over time')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 3. Interpretation (Q10)

*(Fill in after running the notebook)*

Look for:
- **Regime transitions aligned with controller mode changes** → timer-induced liquid level recovery.
- **Gradual drift within a cluster** → progressive valve wear or paraffin build-up.
- **Abrupt switches** → sudden mechanical failure or workover.
- **Diurnal patterns** → if the timer is set to run during off-peak electricity hours.

Hypotheses to test:
1. After a shutdown, the liquid level recovers → first cycles show high fillage (rectangular card).
2. As pumping continues, liquid level drops → gas interference increases → card area shrinks.
3. Long shutdowns (workover) reset the mechanical condition.